# Input Dataset


In [48]:
import pandas as pd
import numpy as np

ufc = pd.read_csv("ufc-clean.csv")

print(ufc.columns)

Index(['date', 'R_fighter', 'B_fighter', 'Winner', 'finish', 'title_bout',
       'weight_class', 'gender', 'no_of_rounds', 'R_current_lose_streak',
       'R_current_win_streak', 'R_draw', 'R_avg_SIG_STR_landed',
       'R_avg_SIG_STR_pct', 'R_avg_SUB_ATT', 'R_avg_TD_landed', 'R_avg_TD_pct',
       'R_longest_win_streak', 'R_losses', 'R_total_rounds_fought',
       'R_total_title_bouts', 'R_win_by_Decision', 'R_win_by_KO_TKO',
       'R_win_by_Submission', 'R_wins', 'R_Stance', 'R_Height_cms',
       'R_Reach_cms', 'R_Weight_lbs', 'R_age', 'B_current_lose_streak',
       'B_current_win_streak', 'B_draw', 'B_avg_SIG_STR_landed',
       'B_avg_SIG_STR_pct', 'B_avg_SUB_ATT', 'B_avg_TD_landed', 'B_avg_TD_pct',
       'B_longest_win_streak', 'B_losses', 'B_total_rounds_fought',
       'B_total_title_bouts', 'B_win_by_Decision', 'B_win_by_KO_TKO',
       'B_win_by_Submission', 'B_wins', 'B_Stance', 'B_Height_cms',
       'B_Reach_cms', 'B_Weight_lbs', 'B_age', 'finish_details',
       'fini

In [ ]:
import pandas as pd
import numpy as np

# Load data
ufc = pd.read_csv("ufc-clean.csv")

# Preserve original row order only as a tie-breaker
ufc["_row_order"] = np.arange(len(ufc))

# Date format example: 3/14/2026
ufc["date"] = pd.to_datetime(
    ufc["date"],
    format="%m/%d/%Y",
    errors="coerce"
)


def safe_divide(num, denom):
    return np.where((denom > 0) & (~pd.isna(denom)), num / denom, 0)


def extract_corner(df, prefix):
    """
    Converts R_ or B_ fighter columns into common fighter-level columns.
    Keeps fight-level timing columns so we can calculate fight-duration metrics.
    """
    corner_cols = [c for c in df.columns if c.startswith(f"{prefix}_")]

    fight_level_cols = [
        "_row_order",
        "date",
        "no_of_rounds",
        "finish_details",
        "finish_round",
        "finish_round_time",
        "total_fight_time_secs"
    ]

    fight_level_cols = [c for c in fight_level_cols if c in df.columns]

    out = df[fight_level_cols + corner_cols].copy()

    rename_map = {}

    for col in corner_cols:
        if col == f"{prefix}_fighter":
            rename_map[col] = "fighter"
        else:
            rename_map[col] = col.replace(f"{prefix}_", "", 1)

    out = out.rename(columns=rename_map)
    out["corner"] = prefix

    return out


# -------------------------------------------------------
# Stack Red and Blue fighters into one fighter-level table
# -------------------------------------------------------

red_fighters = extract_corner(ufc, "R")
blue_fighters = extract_corner(ufc, "B")

fighter_rows = pd.concat(
    [red_fighters, blue_fighters],
    ignore_index=True,
    sort=False
)


# -------------------------------------------------------
# Create fight-duration features from all fighter appearances
# -------------------------------------------------------

fighter_rows["scheduled_fight_time_secs"] = (
    fighter_rows["no_of_rounds"] * 5 * 60
)

fighter_rows["fight_time_pct"] = safe_divide(
    fighter_rows["total_fight_time_secs"],
    fighter_rows["scheduled_fight_time_secs"]
)

fight_time_features = (
    fighter_rows
    .groupby("fighter", as_index=False)
    .agg(
        avg_fight_time_secs=("total_fight_time_secs", "mean"),
        avg_fight_time_pct=("fight_time_pct", "mean")
    )
)

fight_time_features["finish_quickness_score"] = (
    1 - fight_time_features["avg_fight_time_pct"]
)


# -------------------------------------------------------
# Sort by date first, then keep each fighter's most recent stat row
# -------------------------------------------------------

fighter_rows = fighter_rows.sort_values(
    by=["date", "_row_order"],
    ascending=[False, True],
    kind="mergesort",
    na_position="last"
)

latest_fighters = fighter_rows.drop_duplicates(
    subset="fighter",
    keep="first"
).copy()


# -------------------------------------------------------
# Create percentage/rate-based career features
# -------------------------------------------------------

latest_fighters["total_fights"] = (
    latest_fighters["wins"].fillna(0)
    + latest_fighters["losses"].fillna(0)
    + latest_fighters["draw"].fillna(0)
)

latest_fighters["pct_wins_by_decision"] = safe_divide(
    latest_fighters["win_by_Decision"],
    latest_fighters["wins"]
)

latest_fighters["pct_wins_by_ko_tko"] = safe_divide(
    latest_fighters["win_by_KO_TKO"],
    latest_fighters["wins"]
)

latest_fighters["pct_wins_by_submission"] = safe_divide(
    latest_fighters["win_by_Submission"],
    latest_fighters["wins"]
)

latest_fighters["avg_rounds_per_fight"] = safe_divide(
    latest_fighters["total_rounds_fought"],
    latest_fighters["total_fights"]
)


# -------------------------------------------------------
# Add aggregated fight-duration features
# -------------------------------------------------------

latest_fighters = latest_fighters.merge(
    fight_time_features,
    on="fighter",
    how="left"
)


# -------------------------------------------------------
# Drop irrelevant/raw columns
# -------------------------------------------------------

cols_to_drop = [
    "_row_order",
    "date",
    "corner",

    # Fight-level columns used only for timing calculations
    "no_of_rounds",
    "finish_details",
    "finish_round",
    "finish_round_time",
    "total_fight_time_secs",
    "scheduled_fight_time_secs",
    "fight_time_pct",

    # Streak/count outcomes
    "current_lose_streak",
    "current_win_streak",
    "longest_win_streak",
    "draw",
    "wins",
    "losses",
    "total_fights",

    # Raw counts converted into percentages
    "win_by_Decision",
    "win_by_KO_TKO",
    "win_by_Submission",

    # Raw round count converted into average
    "total_rounds_fought",

    # Demographic / physical columns to remove
    "Stance",
    "Weight_lbs",
    "age"
]

fighter_style_stats = latest_fighters.drop(
    columns=[c for c in cols_to_drop if c in latest_fighters.columns]
)

fighter_style_stats = fighter_style_stats.set_index("fighter")

# Save final fighter-level dataset
fighter_style_stats.to_csv("ufc-fighter-style-stats.csv")

fighter_style_stats.head()

,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct,total_title_bouts,Height_cms,Reach_cms,pct_wins_by_decision,pct_wins_by_ko_tko,pct_wins_by_submission,avg_rounds_per_fight,avg_fight_time_secs,avg_fight_time_pct,finish_quickness_score
fighter,,,,,,,,,,,,,,,
Eric Schafer,15.6667,0.588,1.3333,0.8333,0.145,0,190.50,190.50,0.000000,0.333333,0.666667,1.500000,730.333333,0.811481,0.188519
Jason Brilz,31.6667,0.460,0.6667,1.6667,0.500,0,180.34,180.34,0.500000,0.500000,0.000000,2.666667,474.250000,0.526944,0.473056
Brandon Vera,30.7273,0.573,0.0909,0.5455,0.364,0,190.50,193.04,0.285714,0.571429,0.142857,2.090909,740.500000,0.695111,0.304889
Jon Jones,39.0000,0.478,0.2500,2.5000,0.525,0,193.04,213.36,0.666667,0.000000,0.333333,2.250000,952.000000,0.656173,0.343827
Junior Dos Santos,25.7500,0.538,0.0000,0.0000,0.000,0,193.04,195.58,0.000000,1.000000,0.000000,1.500000,756.352941,0.595059,0.404941


: 

In [2]:
import pandas as pd
import numpy as np

ufc = pd.read_csv("ufc-clean.csv")

# ---------- Red fighter engineered features ----------dd
ufc["R_BMI"] = (
    ufc["R_Weight_lbs"] * 703 / 
    (ufc["R_Height_cms"] / 2.54) ** 2
)

ufc["R_Ape_Ratio"] = ufc["R_Reach_cms"] / ufc["R_Height_cms"]

ufc["R_PercWinsByDecision"] = np.where(
    ufc["R_wins"] > 0,
    ufc["R_win_by_Decision"] / ufc["R_wins"],
    0
)

ufc["R_PercWinsByKO"] = np.where(
    ufc["R_wins"] > 0,
    ufc["R_win_by_KO_TKO"] / ufc["R_wins"],
    0
)

ufc["R_PercWinsBySubmission"] = np.where(
    ufc["R_wins"] > 0,
    ufc["R_win_by_Submission"] / ufc["R_wins"],
    0
)

# ---------- Blue fighter engineered features ----------
ufc["B_BMI"] = (
    ufc["B_Weight_lbs"] * 703 / 
    (ufc["B_Height_cms"] / 2.54) ** 2
)

ufc["B_Ape_Ratio"] = ufc["B_Reach_cms"] / ufc["B_Height_cms"]

ufc["B_PercWinsByDecision"] = np.where(
    ufc["B_wins"] > 0,
    ufc["B_win_by_Decision"] / ufc["B_wins"],
    0
)

ufc["B_PercWinsByKO"] = np.where(
    ufc["B_wins"] > 0,
    ufc["B_win_by_KO_TKO"] / ufc["B_wins"],
    0
)

ufc["B_PercWinsBySubmission"] = np.where(
    ufc["B_wins"] > 0,
    ufc["B_win_by_Submission"] / ufc["B_wins"],
    0
)

# ---------- Style features ----------
style_base_cols = [
    "avg_SIG_STR_landed",
    "avg_SIG_STR_pct",
    "avg_SUB_ATT",
    "avg_TD_landed",
    "avg_TD_pct",
    "BMI",
    "Ape_Ratio",
    "PercWinsByDecision",
    "PercWinsByKO",
    "PercWinsBySubmission"
]

red = ufc[["R_fighter"] + [f"R_{col}" for col in style_base_cols]].copy()
red.columns = ["fighter"] + style_base_cols

blue = ufc[["B_fighter"] + [f"B_{col}" for col in style_base_cols]].copy()
blue.columns = ["fighter"] + style_base_cols

fighter_style_stats = (
    pd.concat([red, blue], axis=0)
    .groupby("fighter", as_index=False)[style_base_cols]
    .mean()
)

fighter_style_stats = (
    fighter_style_stats
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

style_feature_cols = style_base_cols

display(fighter_style_stats.head())

,fighter,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct,BMI,Ape_Ratio,PercWinsByDecision,PercWinsByKO,PercWinsBySubmission
1,AJ Cunningham,5.940000,0.290000,0.25,0.000000,0.000000,20.085714,1.014286,0.000,0.000,0.000
2,AJ Dobson,4.290000,0.460000,0.30,1.670000,0.750000,24.405142,1.041096,0.125,0.000,0.875
3,AJ Fletcher,3.360000,0.490000,0.90,1.540000,0.350000,24.389796,0.957143,0.000,0.875,0.125
4,Aalon Cruz,8.310000,0.395000,0.00,0.000000,0.000000,19.663387,1.083333,0.000,0.500,0.000
5,Aaron Phillips,7.126667,0.523333,0.70,0.286667,0.166667,19.933837,1.018116,0.000,0.000,0.000


# K-Means

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

RANDOM_STATE = 42
K = 3

scaler = StandardScaler()
scaled_style_features = scaler.fit_transform(fighter_style_stats[style_feature_cols])

kmeans = KMeans(
    n_clusters=K,
    random_state=RANDOM_STATE,
    n_init=20
)

fighter_style_stats["KMeans_Style"] = kmeans.fit_predict(scaled_style_features) + 1

print("Silhouette:", round(silhouette_score(scaled_style_features, fighter_style_stats["KMeans_Style"]), 4))
print("Davies-Bouldin:", round(davies_bouldin_score(scaled_style_features, fighter_style_stats["KMeans_Style"]), 4))
print("Calinski-Harabasz:", round(calinski_harabasz_score(scaled_style_features, fighter_style_stats["KMeans_Style"]), 4))

display(
    fighter_style_stats["KMeans_Style"]
    .value_counts()
    .sort_index()
    .rename("fighter_count")
    .to_frame()
)


Silhouette: 0.1464
Davies-Bouldin: 2.0871
Calinski-Harabasz: 318.6577


,fighter_count
KMeans_Style,
1,403
2,907
3,825


In [8]:
# ---------- Label K-Means clusters ----------
cluster_summary = (
    fighter_style_stats
    .groupby("KMeans_Style")[style_feature_cols]
    .mean()
)

cluster_summary_z = (
    cluster_summary - fighter_style_stats[style_feature_cols].mean()
) / fighter_style_stats[style_feature_cols].std()

# Style scores
grappler_score = (
    cluster_summary_z["avg_TD_landed"]
    + cluster_summary_z["avg_TD_pct"]
    + cluster_summary_z["avg_SUB_ATT"]
    + cluster_summary_z["PercWinsBySubmission"]
)

striker_score = (
    cluster_summary_z["avg_SIG_STR_landed"]
    + cluster_summary_z["avg_SIG_STR_pct"]
    + cluster_summary_z["PercWinsByKO"]
)

point_score = (
    cluster_summary_z["PercWinsByDecision"]
    + cluster_summary_z["avg_SIG_STR_pct"]
    - cluster_summary_z["PercWinsByKO"]
    - cluster_summary_z["PercWinsBySubmission"]
)

# Assign labels
style_labels = {}

grappler_cluster = grappler_score.idxmax()
style_labels[grappler_cluster] = "Grappler"

remaining = [c for c in cluster_summary.index if c not in style_labels]

striker_cluster = striker_score.loc[remaining].idxmax()
style_labels[striker_cluster] = "Striker"

remaining = [c for c in cluster_summary.index if c not in style_labels]

point_cluster = point_score.loc[remaining].idxmax()
style_labels[point_cluster] = "Point Fighter"

print(style_labels)
display(cluster_summary)


import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA

# 3D PCA visualization
pca = PCA(n_components=3, random_state=RANDOM_STATE)
pca_components = pca.fit_transform(scaled_style_features)

pca_data = pd.DataFrame(
    pca_components,
    columns=["PC1", "PC2", "PC3"]
)

pca_data["fighter"] = fighter_style_stats["fighter"].values
pca_data["KMeans_Style"] = (
    fighter_style_stats["KMeans_Style"]
    .map(style_labels)
    .values
)

fig = px.scatter_3d(
    pca_data,
    x="PC1",
    y="PC2",
    z="PC3",
    color="KMeans_Style",
    hover_name="fighter",
    title="Fighter Style Clustering (3D PCA)"
)

fig.show()

{np.int32(1): 'Grappler', np.int32(2): 'Striker', np.int32(3): 'Point Fighter'}


,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct,BMI,Ape_Ratio,PercWinsByDecision,PercWinsByKO,PercWinsBySubmission
KMeans_Style,,,,,,,,,,
1,12.468144,0.468280,1.392571,1.976525,0.408304,22.696688,1.024460,0.167325,0.112677,0.518587
2,14.689870,0.427171,0.208229,0.518460,0.162647,23.945550,1.025978,0.120373,0.357074,0.033164
3,21.634047,0.458926,0.346540,1.879503,0.423776,22.345463,1.020829,0.561154,0.137857,0.052433


In [9]:
from sklearn.cluster import SpectralClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import pandas as pd
import plotly.express as px

# ---------- Spectral Clustering ----------
K = 3

spectral = SpectralClustering(
    n_clusters=K,
    affinity="nearest_neighbors",
    n_neighbors=10,
    random_state=RANDOM_STATE,
    assign_labels="kmeans"
)

fighter_style_stats["Spectral_Style"] = spectral.fit_predict(scaled_style_features) + 1

print("Silhouette:", round(silhouette_score(scaled_style_features, fighter_style_stats["Spectral_Style"]), 4))
print("Davies-Bouldin:", round(davies_bouldin_score(scaled_style_features, fighter_style_stats["Spectral_Style"]), 4))
print("Calinski-Harabasz:", round(calinski_harabasz_score(scaled_style_features, fighter_style_stats["Spectral_Style"]), 4))

display(
    fighter_style_stats["Spectral_Style"]
    .value_counts()
    .sort_index()
    .rename("fighter_count")
    .to_frame()
)



Silhouette: 0.1174
Davies-Bouldin: 2.1081
Calinski-Harabasz: 254.1795


,fighter_count
Spectral_Style,
1,448
2,1309
3,378


In [10]:
# ---------- Label Spectral clusters ----------
spectral_summary = (
    fighter_style_stats
    .groupby("Spectral_Style")[style_feature_cols]
    .mean()
)

spectral_summary_z = (
    spectral_summary - fighter_style_stats[style_feature_cols].mean()
) / fighter_style_stats[style_feature_cols].std()

grappler_score = (
    spectral_summary_z["avg_TD_landed"]
    + spectral_summary_z["avg_TD_pct"]
    + spectral_summary_z["avg_SUB_ATT"]
    + spectral_summary_z["PercWinsBySubmission"]
)

striker_score = (
    spectral_summary_z["avg_SIG_STR_landed"]
    + spectral_summary_z["avg_SIG_STR_pct"]
    + spectral_summary_z["PercWinsByKO"]
)

point_score = (
    spectral_summary_z["PercWinsByDecision"]
    + spectral_summary_z["avg_SIG_STR_pct"]
    - spectral_summary_z["PercWinsByKO"]
    - spectral_summary_z["PercWinsBySubmission"]
)

spectral_style_labels = {}

grappler_cluster = grappler_score.idxmax()
spectral_style_labels[grappler_cluster] = "Grappler"

remaining = [c for c in spectral_summary.index if c not in spectral_style_labels]

striker_cluster = striker_score.loc[remaining].idxmax()
spectral_style_labels[striker_cluster] = "Striker"

remaining = [c for c in spectral_summary.index if c not in spectral_style_labels]

point_cluster = point_score.loc[remaining].idxmax()
spectral_style_labels[point_cluster] = "Point Fighter"

print(spectral_style_labels)
display(spectral_summary)

# ---------- 3D PCA visualization for Spectral Clustering ----------
pca = PCA(n_components=3, random_state=RANDOM_STATE)
pca_components = pca.fit_transform(scaled_style_features)

spectral_pca_data = pd.DataFrame(
    pca_components,
    columns=["PC1", "PC2", "PC3"]
)

spectral_pca_data["fighter"] = fighter_style_stats["fighter"].values

spectral_pca_data["Spectral_Style"] = (
    fighter_style_stats["Spectral_Style"]
    .map(spectral_style_labels)
    .values
)

fig = px.scatter_3d(
    spectral_pca_data,
    x="PC1",
    y="PC2",
    z="PC3",
    color="Spectral_Style",
    hover_name="fighter",
    title="Fighter Style Clustering - Spectral Clustering 3D PCA"
)

fig.show()

{np.int32(1): 'Grappler', np.int32(2): 'Striker', np.int32(3): 'Point Fighter'}


,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct,BMI,Ape_Ratio,PercWinsByDecision,PercWinsByKO,PercWinsBySubmission
Spectral_Style,,,,,,,,,,
1,13.406792,0.467045,1.302398,1.960769,0.397061,22.551015,1.023518,0.225819,0.097605,0.498600
2,19.874102,0.465640,0.286143,1.331256,0.336769,23.251261,1.023810,0.408451,0.332529,0.043315
3,11.045028,0.359834,0.206165,0.519401,0.113674,23.178916,1.023542,0.009873,0.010582,0.005968


In [11]:
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import pandas as pd
import plotly.express as px

# ---------- GMM Clustering ----------
K = 3

gmm = GaussianMixture(
    n_components=K,
    covariance_type="full",
    random_state=RANDOM_STATE
)

fighter_style_stats["GMM_Style"] = gmm.fit_predict(scaled_style_features) + 1

gmm_probs = gmm.predict_proba(scaled_style_features)

for i in range(K):
    fighter_style_stats[f"GMM_Style_{i+1}_Prob"] = gmm_probs[:, i]

print("BIC:", round(gmm.bic(scaled_style_features), 4))
print("AIC:", round(gmm.aic(scaled_style_features), 4))
print("Silhouette:", round(silhouette_score(scaled_style_features, fighter_style_stats["GMM_Style"]), 4))
print("Davies-Bouldin:", round(davies_bouldin_score(scaled_style_features, fighter_style_stats["GMM_Style"]), 4))
print("Calinski-Harabasz:", round(calinski_harabasz_score(scaled_style_features, fighter_style_stats["GMM_Style"]), 4))

display(
    fighter_style_stats["GMM_Style"]
    .value_counts()
    .sort_index()
    .rename("fighter_count")
    .to_frame()
)

BIC: 29905.1212
AIC: 28788.8754
Silhouette: 0.0843
Davies-Bouldin: 2.7938
Calinski-Harabasz: 181.0612


,fighter_count
GMM_Style,
1,790
2,736
3,609


In [12]:
# ---------- Label GMM clusters ----------
gmm_summary = (
    fighter_style_stats
    .groupby("GMM_Style")[style_feature_cols]
    .mean()
)

gmm_summary_z = (
    gmm_summary - fighter_style_stats[style_feature_cols].mean()
) / fighter_style_stats[style_feature_cols].std()

grappler_score = (
    gmm_summary_z["avg_TD_landed"]
    + gmm_summary_z["avg_TD_pct"]
    + gmm_summary_z["avg_SUB_ATT"]
    + gmm_summary_z["PercWinsBySubmission"]
)

striker_score = (
    gmm_summary_z["avg_SIG_STR_landed"]
    + gmm_summary_z["avg_SIG_STR_pct"]
    + gmm_summary_z["PercWinsByKO"]
)

point_score = (
    gmm_summary_z["PercWinsByDecision"]
    + gmm_summary_z["avg_SIG_STR_pct"]
    - gmm_summary_z["PercWinsByKO"]
    - gmm_summary_z["PercWinsBySubmission"]
)

gmm_style_labels = {}

grappler_cluster = grappler_score.idxmax()
gmm_style_labels[grappler_cluster] = "Grappler"

remaining = [c for c in gmm_summary.index if c not in gmm_style_labels]

striker_cluster = striker_score.loc[remaining].idxmax()
gmm_style_labels[striker_cluster] = "Striker"

remaining = [c for c in gmm_summary.index if c not in gmm_style_labels]

point_cluster = point_score.loc[remaining].idxmax()
gmm_style_labels[point_cluster] = "Point Fighter"

print(gmm_style_labels)
display(gmm_summary)

# ---------- 3D PCA visualization for GMM ----------
pca = PCA(n_components=3, random_state=RANDOM_STATE)
pca_components = pca.fit_transform(scaled_style_features)

gmm_pca_data = pd.DataFrame(
    pca_components,
    columns=["PC1", "PC2", "PC3"]
)

gmm_pca_data["fighter"] = fighter_style_stats["fighter"].values

gmm_pca_data["GMM_Style"] = (
    fighter_style_stats["GMM_Style"]
    .map(gmm_style_labels)
    .values
)

fig = px.scatter_3d(
    gmm_pca_data,
    x="PC1",
    y="PC2",
    z="PC3",
    color="GMM_Style",
    hover_name="fighter",
    title="Fighter Style Clustering - GMM 3D PCA"
)

fig.show()

{np.int64(1): 'Grappler', np.int64(3): 'Striker', np.int64(2): 'Point Fighter'}


,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct,BMI,Ape_Ratio,PercWinsByDecision,PercWinsByKO,PercWinsBySubmission
GMM_Style,,,,,,,,,,
1,16.252479,0.466486,0.905693,1.716852,0.377441,22.895229,1.024290,0.293262,0.207831,0.357377
2,17.126424,0.406760,0.296580,1.156842,0.255694,22.872485,1.021768,0.298211,0.000000,0.000000
3,17.655101,0.471061,0.167792,1.001023,0.287872,23.610847,1.025275,0.309361,0.523516,0.000000


In [13]:
# Numeric style -> label
print(gmm_style_labels)

# Add readable label
fighter_style_stats["GMM_Style_Label"] = fighter_style_stats["GMM_Style"].map(gmm_style_labels)

# Show average stats for each GMM style
gmm_summary_labeled = gmm_summary.copy()
gmm_summary_labeled["Style_Label"] = gmm_summary_labeled.index.map(gmm_style_labels)

display(gmm_summary_labeled)

{np.int64(1): 'Grappler', np.int64(3): 'Striker', np.int64(2): 'Point Fighter'}


,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct,BMI,Ape_Ratio,PercWinsByDecision,PercWinsByKO,PercWinsBySubmission,Style_Label
GMM_Style,,,,,,,,,,,
1,16.252479,0.466486,0.905693,1.716852,0.377441,22.895229,1.024290,0.293262,0.207831,0.357377,Grappler
2,17.126424,0.406760,0.296580,1.156842,0.255694,22.872485,1.021768,0.298211,0.000000,0.000000,Point Fighter
3,17.655101,0.471061,0.167792,1.001023,0.287872,23.610847,1.025275,0.309361,0.523516,0.000000,Striker


In [14]:
def search_fighter_style(name, model="GMM"):
    matches = fighter_style_stats[
        fighter_style_stats["fighter"].str.contains(name, case=False, na=False)
    ].copy()

    if matches.empty:
        print("No fighter found.")
        return

    if model == "GMM":
        # Add readable label if labels exist
        if "gmm_style_labels" in globals():
            fighter_style_stats["GMM_Style_Label"] = fighter_style_stats["GMM_Style"].map(gmm_style_labels)
            matches["GMM_Style_Label"] = matches["GMM_Style"].map(gmm_style_labels)

        prob_cols = [
            col for col in fighter_style_stats.columns
            if col.startswith("GMM_Style_") and col.endswith("_Prob")
        ]

        output_cols = ["fighter", "GMM_Style"]

        if "GMM_Style_Label" in fighter_style_stats.columns:
            output_cols.append("GMM_Style_Label")

        output_cols += prob_cols

        display(matches[output_cols].sort_values("fighter"))

    elif model == "KMeans":
        if "style_labels" in globals():
            fighter_style_stats["KMeans_Style_Label"] = fighter_style_stats["KMeans_Style"].map(style_labels)
            matches["KMeans_Style_Label"] = matches["KMeans_Style"].map(style_labels)

        output_cols = ["fighter", "KMeans_Style"]

        if "KMeans_Style_Label" in fighter_style_stats.columns:
            output_cols.append("KMeans_Style_Label")

        display(matches[output_cols].sort_values("fighter"))

    elif model == "Spectral":
        if "spectral_style_labels" in globals():
            fighter_style_stats["Spectral_Style_Label"] = fighter_style_stats["Spectral_Style"].map(spectral_style_labels)
            matches["Spectral_Style_Label"] = matches["Spectral_Style"].map(spectral_style_labels)

        output_cols = ["fighter", "Spectral_Style"]

        if "Spectral_Style_Label" in fighter_style_stats.columns:
            output_cols.append("Spectral_Style_Label")

        display(matches[output_cols].sort_values("fighter"))

    else:
        print("Model must be one of: 'GMM', 'KMeans', 'Spectral'")

In [15]:
search_fighter_style("thompson", model="Spectral")

,fighter,Spectral_Style,Spectral_Style_Label
1646,Oli Thompson,3,Point Fighter
1974,Stephen Thompson,2,Striker


In [18]:

import hdbscan
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

hdb = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=3
)

fighter_style_stats["HDBSCAN_Style"] = hdb.fit_predict(scaled_style_features)

print("Cluster counts:")
display(
    fighter_style_stats["HDBSCAN_Style"]
    .value_counts()
    .sort_index()
    .rename("fighter_count")
    .to_frame()
)

# Evaluate only non-noise fighters
non_noise_mask = fighter_style_stats["HDBSCAN_Style"] != -1
hdb_labels_non_noise = fighter_style_stats.loc[non_noise_mask, "HDBSCAN_Style"]
hdb_features_non_noise = scaled_style_features[non_noise_mask]

if hdb_labels_non_noise.nunique() > 1:
    print("Silhouette:", round(silhouette_score(hdb_features_non_noise, hdb_labels_non_noise), 4))
    print("Davies-Bouldin:", round(davies_bouldin_score(hdb_features_non_noise, hdb_labels_non_noise), 4))
    print("Calinski-Harabasz:", round(calinski_harabasz_score(hdb_features_non_noise, hdb_labels_non_noise), 4))
else:
    print("Not enough non-noise clusters for clustering metrics.")

Cluster counts:


,fighter_count
HDBSCAN_Style,
-1,1614
0,5
1,6
2,5
3,5
4,9
5,5
6,5
7,21


Silhouette: 0.0749
Davies-Bouldin: 1.243
Calinski-Harabasz: 40.9099


In [27]:
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import pandas as pd

# ============================================================
# Hybrid preprocessing + regular GMM
# ============================================================

RANDOM_STATE = 42
K = 6

continuous_cols = [
    "avg_SIG_STR_landed",
    "avg_SIG_STR_pct",
    "avg_SUB_ATT",
    "avg_TD_landed",
    "avg_TD_pct",
    "BMI",
    "Ape_Ratio",
    "PercWinsByDecision",
    "PercWinsByKO",
    "PercWinsBySubmission"
]

count_cols = [
    "wins",
    "losses",
    "win_by_Decision",
    "win_by_KO_TKO",
    "win_by_Submission",
    "total_rounds_fought",
    "total_title_bouts",
    "longest_win_streak"
]

# ---------- Build fighter-level table with both feature types ----------
red_cols = ["R_fighter"] + [f"R_{col}" for col in continuous_cols + count_cols]
blue_cols = ["B_fighter"] + [f"B_{col}" for col in continuous_cols + count_cols]

red = ufc[red_cols].copy()
red.columns = ["fighter"] + continuous_cols + count_cols

blue = ufc[blue_cols].copy()
blue.columns = ["fighter"] + continuous_cols + count_cols

fighter_hybrid_stats = (
    pd.concat([red, blue], axis=0)
    .groupby("fighter", as_index=False)[continuous_cols + count_cols]
    .mean()
)

fighter_hybrid_stats = (
    fighter_hybrid_stats
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

# ---------- Transform each block differently ----------
X_cont = fighter_hybrid_stats[continuous_cols].copy()
X_count = fighter_hybrid_stats[count_cols].copy()

X_count_log = np.log1p(X_count.clip(lower=0))

X_cont_scaled = StandardScaler().fit_transform(X_cont)
X_count_scaled = StandardScaler().fit_transform(X_count_log)

X_hybrid = np.hstack([X_cont_scaled, X_count_scaled])

# ---------- Regular GMM ----------
gmm_hybrid = GaussianMixture(
    n_components=K,
    covariance_type="full",
    random_state=RANDOM_STATE
)

fighter_hybrid_stats["GMM_Hybrid_Style"] = gmm_hybrid.fit_predict(X_hybrid) + 1

gmm_hybrid_probs = gmm_hybrid.predict_proba(X_hybrid)

for i in range(K):
    fighter_hybrid_stats[f"GMM_Hybrid_Style_{i+1}_Prob"] = gmm_hybrid_probs[:, i]

# ---------- Metrics ----------
print("BIC:", round(gmm_hybrid.bic(X_hybrid), 4))
print("AIC:", round(gmm_hybrid.aic(X_hybrid), 4))
print("Silhouette:", round(silhouette_score(X_hybrid, fighter_hybrid_stats["GMM_Hybrid_Style"]), 4))
print("Davies-Bouldin:", round(davies_bouldin_score(X_hybrid, fighter_hybrid_stats["GMM_Hybrid_Style"]), 4))
print("Calinski-Harabasz:", round(calinski_harabasz_score(X_hybrid, fighter_hybrid_stats["GMM_Hybrid_Style"]), 4))

display(
    fighter_hybrid_stats["GMM_Hybrid_Style"]
    .value_counts()
    .sort_index()
    .rename("fighter_count")
    .to_frame()
)

display(
    fighter_hybrid_stats
    .groupby("GMM_Hybrid_Style")[continuous_cols + count_cols]
    .mean()
)

BIC: -14674.8853
AIC: -21128.7121
Silhouette: 0.0968
Davies-Bouldin: 2.2347
Calinski-Harabasz: 244.4074


,fighter_count
GMM_Hybrid_Style,
1,172
2,421
3,302
4,508
5,335
6,397


,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct,BMI,Ape_Ratio,PercWinsByDecision,PercWinsByKO,PercWinsBySubmission,wins,losses,win_by_Decision,win_by_KO_TKO,win_by_Submission,total_rounds_fought,total_title_bouts,longest_win_streak
GMM_Hybrid_Style,,,,,,,,,,,,,,,,,,
1,11.822096,0.469198,1.359014,1.683208,0.363209,23.039836,1.025042,0.000000,0.177550,0.560110,1.395036,1.219807,0.000000,0.441304,0.943446,4.936451,0.030233,1.097576
2,12.346278,0.383350,0.244818,0.854500,0.204944,23.178558,1.022698,0.000000,0.039192,0.000000,0.039192,0.713579,0.000000,0.039192,0.000000,1.646437,0.000000,0.039192
3,14.564996,0.463276,0.889225,1.923848,0.393540,22.046967,1.019703,0.414613,0.061753,0.360225,2.713625,1.535097,1.377864,0.236167,1.066100,10.235514,0.116806,1.986002
4,15.861340,0.476281,0.190309,1.078935,0.288383,23.547005,1.025384,0.306947,0.526069,0.000000,2.267547,1.418178,0.997383,1.242911,0.000000,8.470989,0.000000,1.654837
5,22.973610,0.436646,0.348868,1.502440,0.319094,22.473686,1.020809,0.641365,0.000000,0.000000,1.034882,0.928879,1.034882,0.000000,0.000000,5.416202,0.069655,0.882853
6,22.198818,0.464853,0.546733,1.349365,0.354371,23.754693,1.027514,0.361617,0.378025,0.194461,6.158615,3.039428,2.460099,2.363805,1.285975,21.954057,1.036542,3.483513


In [31]:
# Add labels first
gmm_hybrid_style_labels = {
    1: "Submission Grappler",
    2: "Low-Data / Inexperienced Fighter",
    3: "Grappling Decision Fighter",
    4: "Power Striker",
    5: "Decision / Point Fighter",
    6: "Elite Veteran All-Rounder"
}

fighter_hybrid_stats["GMM_Hybrid_Style_Label"] = (
    fighter_hybrid_stats["GMM_Hybrid_Style"].map(gmm_hybrid_style_labels)
)
# Search function with labels included
def search_hybrid_gmm_style(name):
    prob_cols = [
        col for col in fighter_hybrid_stats.columns
        if col.startswith("GMM_Hybrid_Style_") and col.endswith("_Prob")
    ]

    matches = fighter_hybrid_stats[
        fighter_hybrid_stats["fighter"].str.contains(name, case=False, na=False)
    ].copy()

    if matches.empty:
        print("No fighter found.")
        return

    display(
        matches[
            ["fighter", "GMM_Hybrid_Style", "GMM_Hybrid_Style_Label"] + prob_cols
        ].sort_values("fighter")
    )

In [36]:
search_hybrid_gmm_style("Khabib")

,fighter,GMM_Hybrid_Style,GMM_Hybrid_Style_Label,GMM_Hybrid_Style_1_Prob,GMM_Hybrid_Style_2_Prob,GMM_Hybrid_Style_3_Prob,GMM_Hybrid_Style_4_Prob,GMM_Hybrid_Style_5_Prob,GMM_Hybrid_Style_6_Prob
1226,Khabib Nurmagomedov,6,Elite Veteran All-Rounder,0.0,0.0,0.000001,0.0,0.0,0.999999
